# PCam fine-tune on Colab T4

Fine-tune a pretrained ResNet-18 on the Kaggle Histopathologic Cancer Detection
(PCam) dataset. Target val AUC ≈ 0.94–0.96 in 2 epochs.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

**Also:** accept the competition rules once at
<https://www.kaggle.com/c/histopathologic-cancer-detection/rules>,
otherwise the Kaggle API refuses to download the data.

## Cell 1 — GPU check + Kaggle auth

In [1]:
!nvidia-smi
from google.colab import files
print("kaggle.json dosyanı yükle:")
files.upload()

!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install timm

Sat Apr 18 15:41:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Saving kaggle.json to kaggle.json


## Cell 2 — Download dataset (~7 GB)

In [2]:
!kaggle competitions download -c histopathologic-cancer-detection
!unzip -q histopathologic-cancer-detection.zip -d pcam
!ls pcam | head

100% 6.31G/6.31G [01:16<00:00, 89.0MB/s]

sample_submission.csv
test
train
train_labels.csv


## Cell 3 — Data pipeline

In [3]:
import torch, pandas as pd, numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA = Path('pcam'); TRAIN = DATA / 'train'
device = torch.device('cuda')

df = pd.read_csv(DATA / 'train_labels.csv')
train_df, val_df = train_test_split(df, test_size=0.1, stratify=df.label, random_state=42)
print(f"train={len(train_df)}, val={len(val_df)}, pozitif oran={df.label.mean():.3f}")

IM_MEAN, IM_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(15),
    T.ColorJitter(0.1, 0.1, 0.1), T.ToTensor(), T.Normalize(IM_MEAN, IM_STD),
])
val_tf = T.Compose([T.ToTensor(), T.Normalize(IM_MEAN, IM_STD)])

class PCam(Dataset):
    def __init__(self, df, tf): self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(TRAIN / f'{r.id}.tif').convert('RGB')
        return self.tf(img), torch.tensor(r.label, dtype=torch.float32)

train_loader = DataLoader(PCam(train_df, train_tf), batch_size=256,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(PCam(val_df, val_tf), batch_size=512,
                          shuffle=False, num_workers=2, pin_memory=True)

train=198022, val=22003, pozitif oran=0.405


## Cell 4 — Model + training

In [4]:
import timm, torch.nn as nn
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

model = timm.create_model('resnet18', pretrained=True, num_classes=1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
crit = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler()

EPOCHS = 2
for ep in range(EPOCHS):
    model.train(); total = 0
    for x, y in tqdm(train_loader, desc=f'epoch {ep+1} train'):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(x).squeeze(-1)
            loss = crit(logits, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        total += loss.item()

    model.eval(); preds, labs = [], []
    with torch.no_grad(), torch.cuda.amp.autocast():
        for x, y in tqdm(val_loader, desc=f'epoch {ep+1} val'):
            logits = model(x.to(device)).squeeze(-1)
            preds.extend(torch.sigmoid(logits).float().cpu().numpy())
            labs.extend(y.numpy())
    print(f'epoch {ep+1}: train_loss={total/len(train_loader):.4f}, val_AUC={roc_auc_score(labs, preds):.4f}')

torch.save(model.state_dict(), 'resnet18_pcam.pth')
files.download('resnet18_pcam.pth')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

/tmp/ipykernel_3580/3230781416.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


epoch 1 train:   0%|          | 0/774 [00:00<?, ?it/s]

/tmp/ipykernel_3580/3230781416.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_3580/3230781416.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


...

epoch 2 val:   0%|          | 0/43 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a5a60f64d60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Exception ignored in:     self._shutdown_workers()
<function _MultiProcessingDataLoaderIter.__del__ at 0x7a5a60f64d60>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

Traceback (most recent call last):
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
if w.is_alive():
    self._shutdown_workers() 
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     if w.is_alive(): 
     ^  ^  ^^^^^^^^^^^^^^^^^^
^Exception ignored in: ^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
<function _MultiProcessingDataLoaderIter.__del__ at 0x7a5a60f64d60>^    ^
assert

epoch 2: train_loss=0.2240, val_AUC=0.9776


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>